# First-contact emitter assignment pipeline

This notebook runs the end-to-end demonstration requested for `LuaHistory_2026-06-23.txt`: start from the first CMO `PY_CONTACT_LOG` emission line, normalize it into the repository observation ontology, recruit the knowledge graph to generate a user-configurable number of candidate emitter-platform/operator hypotheses, extract graph-neighbourhood features, produce a probabilistic assignment for the emitter platform type and operator nation, and prepare an LLM explanation-layer payload that explains—but does not change—the model probabilities.

The notebook is designed to be runnable in two modes:

1. **Offline deterministic demo**: uses the local parser plus a small auditable candidate-reference table derived from the first emission sensor name (`Slot Back [N-010 Zhuk-M]`). This lets the whole flow run without Neo4j credentials while preserving the same hypothesis contract used by graph-backed runs.
2. **Neo4j-backed run** (default): keep `USE_NEO4J = True` and provide credentials so candidate hypotheses are generated from knowledge-graph evidence about radar aliases, platform classes, operators, and aircraft kinematics.

The country/nationality distribution below is explicitly an **operator nation** distribution. It is not the platform `country_of_origin`, because exported aircraft may be operated by nations other than their design/manufacturing origin.



In [1]:
from __future__ import annotations

import json
import math
import os
import re
from dataclasses import asdict
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "LuaHistory_2026-06-23.txt").exists() and (REPO_ROOT.parent / "LuaHistory_2026-06-23.txt").exists():
    REPO_ROOT = REPO_ROOT.parent

LUA_HISTORY = REPO_ROOT / "LuaHistory_2026-06-23.txt"
WORK_DIR = REPO_ROOT / "notebooks" / "outputs" / "first_contact_emitter_assignment"
WORK_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repository: {REPO_ROOT}")
print(f"Input log:  {LUA_HISTORY}")
print(f"Outputs:    {WORK_DIR}")


Repository: C:\Users\theon\CMO-Sensor-Fusion
Input log:  C:\Users\theon\CMO-Sensor-Fusion\LuaHistory_2026-06-23.txt
Outputs:    C:\Users\theon\CMO-Sensor-Fusion\notebooks\outputs\first_contact_emitter_assignment


## 1. Read the first contact emission

The first line in the provided Lua history file is the first contact emission record. We keep its source line number for provenance and downstream graph audit IDs.


In [2]:
from combat_id_calibration.cmo_observation_ingest import parse_observation_line, write_observations_jsonl

first_line_number = None
first_line = None
with LUA_HISTORY.open(encoding="utf-8", errors="replace") as handle:
    for line_number, line in enumerate(handle, start=1):
        if "PY_CONTACT_LOG" in line:
            first_line_number = line_number
            first_line = line.strip()
            break

if first_line is None:
    raise RuntimeError(f"No PY_CONTACT_LOG line found in {LUA_HISTORY}")

observation = parse_observation_line(first_line, source_line=first_line_number)
if observation is None:
    raise RuntimeError("The first PY_CONTACT_LOG line could not be parsed")

observation_record = asdict(observation)
observations_jsonl = WORK_DIR / "first_contact_observation.jsonl"
write_observations_jsonl([observation], observations_jsonl)

print(f"First emission source line: {first_line_number}")
print(json.dumps(observation_record, indent=2, sort_keys=True))
print(f"Wrote {observations_jsonl.relative_to(REPO_ROOT)}")


First emission source line: 1
{
  "emission_age": 33.900054931641,
  "emission_altitude": 10316.349609375,
  "emission_classificationlevel": 2,
  "emission_heading": 331.40036010742,
  "emission_latitude": 44.640933862914,
  "emission_longitude": 31.890743606263,
  "emission_role": 2122,
  "emission_sensor_name": "Slot Back [N-010 Zhuk-M]",
  "emission_solid": true,
  "emission_speed": 479.64691162109,
  "emission_target_type": "Type: Multirole (Fighter/Attack)",
  "emission_type": 2001,
  "observation_id": "13e4329885a9f92e",
  "schema": "cmo_emission_observation_v1",
  "sensor_aircraft": "Typhoon FGR.4",
  "source": "cmo_lua",
  "source_line": 1,
  "time": 1844772240
}
Wrote notebooks\outputs\first_contact_emitter_assignment\first_contact_observation.jsonl


## 2. Optional graph ingestion

Set `USE_NEO4J = True` when a Neo4j service is available. The offline path still creates the same JSONL artifact and then performs deterministic feature extraction locally so the notebook remains reproducible in a clean development environment.


In [3]:
USE_NEO4J = True  # Set to False to use the deterministic offline fallback instead of Neo4j.
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = 'password123'
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE") or None

if USE_NEO4J:
    from combat_id_calibration.cmo_observation_ingest import populate_observations_neo4j
    populate_observations_neo4j([observation], NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD, NEO4J_DATABASE)
    print("Observation ingested into Neo4j")
else:
    print("Offline mode: skipped Neo4j write. JSONL observation artifact is available for later ingestion.")



Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL (obs, contact) { ... }', position=<SummaryInputPosition line=22, column=9, offset=1394>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 1394, 'line': 22, 'column': 9}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n        MERGE (obs:Observation {id: $observation_id})\n          SET obs.schema = $schema, obs.time = $time, obs.source = $source,\n              obs.source_line = $source_line, obs.age_seconds = $emission_age,\n              obs.solid = $emission_solid, obs.latitude = $emission_latitude,\n              obs.longitude = $emi

Observation ingested into Neo4j


## 3. Candidate emitter-platform/operator hypotheses

`HYPOTHESIS_COUNT` is the user-configurable number of candidate hypotheses to produce for the first contact. In Neo4j-backed runs, the helper queries the knowledge graph for platforms linked to the observed emitter aliases, compatible platform classes, kinematic envelopes, and operator relationships; the LLM prompt then asks the model to select exactly `HYPOTHESIS_COUNT` mutually competing hypotheses from that evidence.

The offline candidate table below is a deterministic stand-in for that graph evidence. It intentionally includes `operator_nation` instead of `country_of_origin` so the probability model marginalizes over who operates the aircraft, not where the aircraft was designed or manufactured.


In [ ]:
HYPOTHESIS_COUNT = int(os.getenv("HYPOTHESIS_COUNT", "10"))

# candidate_reference = [
#     {
#         "hypothesis": "MiG-29SMT (Russia-operated)",
#         "operator_nation": "Russia",
#         "emitter_aliases": ["Slot Back", "N-010 Zhuk-M", "Zhuk-M"],
#         "platform_class": "Type: Multirole (Fighter/Attack)",
#         "typical_speed_kt": (250, 850),
#         "typical_altitude_m": (0, 18000),
#     },
#     {
#         "hypothesis": "MiG-29K",
#         "operator_nation": "India",
#         "emitter_aliases": ["Slot Back", "N-010 Zhuk-M", "Zhuk-ME"],
#         "platform_class": "Type: Multirole (Fighter/Attack)",
#         "typical_speed_kt": (220, 850),
#         "typical_altitude_m": (0, 17500),
#     },
#     {
#         "hypothesis": "MiG-29SMT (Yemen-operated)",
#         "operator_nation": "Yemen",
#         "emitter_aliases": ["Slot Back", "N-010 Zhuk-M", "Zhuk-M"],
#         "platform_class": "Type: Multirole (Fighter/Attack)",
#         "typical_speed_kt": (250, 850),
#         "typical_altitude_m": (0, 18000),
#     },
#     {
#         "hypothesis": "Su-30MK",
#         "operator_nation": "Algeria",
#         "emitter_aliases": ["Bars", "N011M"],
#         "platform_class": "Type: Multirole (Fighter/Attack)",
#         "typical_speed_kt": (220, 900),
#         "typical_altitude_m": (0, 18000),
#     },
#     {
#         "hypothesis": "F-16C",
#         "operator_nation": "United States",
#         "emitter_aliases": ["APG-68", "AN/APG-68"],
#         "platform_class": "Type: Multirole (Fighter/Attack)",
#         "typical_speed_kt": (220, 900),
#         "typical_altitude_m": (0, 15000),
#     },
# ]


def observed_emitter_aliases(sensor_name: str) -> list[str]:
    """Return useful alias tokens from a CMO sensor string such as 'Slot Back [N-010 Zhuk-M]'."""
    bracketed = re.findall(r"\[([^\]]+)\]", sensor_name)
    without_brackets = re.sub(r"\s*\[[^\]]+\]", "", sensor_name).strip()
    aliases = [without_brackets, *bracketed, sensor_name]
    aliases.extend(part.strip() for item in bracketed for part in re.split(r"[/,;]", item))
    seen = set()
    return [alias for alias in aliases if alias and not (alias.lower() in seen or seen.add(alias.lower()))]


def graph_hypothesis_query(emitter_aliases: list[str], limit: int) -> tuple[str, dict[str, object]]:
    """Build the parameterized Neo4j hypothesis query and its Cypher parameters."""
    query = """
    MATCH (emitter:Entity)
    WHERE any(alias IN $emitter_aliases WHERE toLower(coalesce(emitter.name, emitter.id, '')) CONTAINS toLower(alias))
       OR EXISTS {
            MATCH (alias_entity:Entity)-[alias_fact:FACT]-(emitter)
            WHERE alias_fact.predicate IN ['ALSO_KNOWN_AS', 'HAS_SENSOR', 'EMITS', 'SUPPORTS_IDENTIFICATION', 'INDICATES']
              AND any(alias IN $emitter_aliases WHERE toLower(coalesce(alias_entity.name, alias_entity.id, alias_fact.evidence, '')) CONTAINS toLower(alias))
       }
    MATCH path = (emitter)-[:FACT*1..3]-(platform:Entity)
    WHERE any(rel IN relationships(path) WHERE rel.predicate IN ['HAS_SENSOR', 'EMITS', 'HAS_SUBSYSTEM', 'SUPPORTS_IDENTIFICATION', 'INDICATES', 'IS_A', 'VARIANT_OF'])
      AND none(node IN nodes(path) WHERE toLower(coalesce(node.name, '')) IN ['radar', 'sensor', 'aircraft radar'])
    OPTIONAL MATCH (platform)-[operator_fact:FACT {predicate: 'OPERATED_BY'}]->(operator:Entity)
    WITH platform, operator, emitter, path,
         [rel IN relationships(path) | coalesce(rel.evidence, '')] AS path_evidence,
         [node IN nodes(path) | coalesce(node.name, node.id)] AS path_nodes
    WITH coalesce(platform.name, platform.id) AS hypothesis,
         coalesce(operator.name, 'unknown') AS operator_nation,
         collect(DISTINCT coalesce(emitter.name, emitter.id)) AS matched_emitters,
         collect(DISTINCT path_nodes)[..5] AS evidence_paths,
         collect(DISTINCT path_evidence)[..5] AS evidence_text,
         count(DISTINCT path) AS support_count
    RETURN hypothesis,
           operator_nation,
           matched_emitters,
           evidence_paths,
           evidence_text,
           support_count
    ORDER BY support_count DESC, hypothesis ASC, operator_nation ASC
    LIMIT $limit
    """.strip()
    return query, {"emitter_aliases": emitter_aliases, "limit": limit}


def fetch_graph_hypotheses(obs, n: int) -> list[dict[str, object]]:
    """Query Neo4j for candidate platform/operator hypotheses grounded in emitter-alias evidence."""
    from combat_id_calibration.graph_ingest import _validate_neo4j_credentials
    import neo4j

    emitter_aliases = observed_emitter_aliases(obs.emission_sensor_name)
    print(emitter_aliases)
    user, password = _validate_neo4j_credentials(NEO4J_USER, NEO4J_PASSWORD)
    driver = neo4j.GraphDatabase.driver(NEO4J_URI, auth=(user, password))
    try:
        driver.verify_connectivity()
        session_kwargs = {"database": NEO4J_DATABASE} if NEO4J_DATABASE else {}
        with driver.session(**session_kwargs) as session:
            query, params = graph_hypothesis_query(emitter_aliases, limit=max(n * 4, n))
            rows = [dict(row) for row in session.run(query, **params)]
    finally:
        driver.close()

    hypotheses = []
    for row in rows:
        hypothesis = str(row.get("hypothesis") or "").strip()
        if not hypothesis:
            continue
        matched_emitters = [str(item) for item in row.get("matched_emitters") or [] if item]
        hypotheses.append({
            "hypothesis": hypothesis,
            "operator_nation": str(row.get("operator_nation") or "unknown"),
            "emitter_aliases": matched_emitters or emitter_aliases,
            "platform_class": obs.emission_target_type,
            "typical_speed_kt": (0, 1200),
            "typical_altitude_m": (0, 25000),
            "kg_support_count": int(row.get("support_count") or 0),
            "kg_evidence_paths": row.get("evidence_paths") or [],
            "kg_evidence_text": row.get("evidence_text") or [],
        })
    return hypotheses[:n]



def build_llm_hypothesis_prompt(obs, kg_rows: list[dict[str, object]], n: int) -> str:
    return "\n".join([
        f"Generate exactly {n} candidate emitter-platform/operator hypotheses from the knowledge-graph rows.",
        "Use both kinematic compatibility and emitter-type/radar-alias compatibility.",
        "Return JSON only: {\"hypotheses\":[{\"hypothesis\": str, \"operator_nation\": str, \"rationale\": str}]}",
        "Do not use country_of_origin as a substitute for operator_nation; exported aircraft may be operated by different nations.",
        f"Observed emitter aliases: {observed_emitter_aliases(obs.emission_sensor_name)}.",
        f"Observed emitter: {obs.emission_sensor_name}; target type: {obs.emission_target_type}; speed kt: {obs.emission_speed}; altitude m: {obs.emission_altitude}.",
        f"Knowledge-graph evidence rows: {json.dumps(kg_rows, sort_keys=True)}",
    ])


def select_offline_hypotheses(obs, candidates: list[dict[str, object]], n: int) -> list[dict[str, object]]:
    def score(candidate: dict[str, object]) -> tuple[float, str, str]:
        aliases = candidate["emitter_aliases"]
        alias_score = sum(1 for alias in aliases if alias.lower() in obs.emission_sensor_name.lower())
        class_score = 1 if obs.emission_target_type == candidate["platform_class"] else 0
        speed_low, speed_high = candidate["typical_speed_kt"]
        alt_low, alt_high = candidate["typical_altitude_m"]
        speed_score = 1 if obs.emission_speed is not None and speed_low <= obs.emission_speed <= speed_high else 0
        alt_score = 1 if obs.emission_altitude is not None and alt_low <= obs.emission_altitude <= alt_high else 0
        kg_score = float(candidate.get("kg_support_count") or 0)
        return (kg_score + alias_score * 3 + class_score + speed_score + alt_score, candidate["hypothesis"], candidate["operator_nation"])

    return sorted(candidates, key=score, reverse=True)[:n]

if USE_NEO4J:
    kg_rows = fetch_graph_hypotheses(observation, HYPOTHESIS_COUNT)
    if len(kg_rows) < HYPOTHESIS_COUNT:
        raise RuntimeError(f"Neo4j returned {len(kg_rows)} candidate hypotheses; expected at least {HYPOTHESIS_COUNT}.")
    candidate_hypotheses = select_offline_hypotheses(observation, kg_rows, HYPOTHESIS_COUNT)
else:
    print('NEO4J AVAILABILITY ISSUE')
llm_hypothesis_prompt = build_llm_hypothesis_prompt(observation, kg_rows, HYPOTHESIS_COUNT)

print(llm_hypothesis_prompt)
print(json.dumps(candidate_hypotheses, indent=2))





## 4. Feature extraction

The repository feature contract is one row per `(scenario_id, contact_id, observation_time, hypothesis)`. In Neo4j mode, you can call `extract_features_neo4j`; in offline mode, the helper below mirrors that contract with transparent matching features from the first emission line.


In [ ]:
from combat_id_calibration.feature_extraction import ContactHypothesisFeatures, evidence_query_id, feature_logit


def _alias_match_score(sensor_name: str, aliases: list[str]) -> float:
    sensor = sensor_name.lower()
    matches = sum(1 for alias in aliases if alias.lower() in sensor)
    return min(1.0, matches / max(1, min(2, len(aliases))))


def _range_score(value: float | None, low: float, high: float) -> float:
    if value is None:
        return 0.0
    return 1.0 if low <= float(value) <= high else 0.2


def offline_features_for_candidate(obs, candidate: dict[str, object]) -> dict[str, object]:
    emission_match = _alias_match_score(obs.emission_sensor_name, candidate["emitter_aliases"])
    class_match = 1.0 if obs.emission_target_type == candidate["platform_class"] else 0.0
    speed_score = _range_score(obs.emission_speed, *candidate["typical_speed_kt"])
    altitude_score = _range_score(obs.emission_altitude, *candidate["typical_altitude_m"])
    kinematic_match = (speed_score + altitude_score) / 2.0
    supporting_paths = (2.0 * emission_match) + class_match + kinematic_match
    contradicting_paths = 0.0 if emission_match else 1.0
    contradiction_score = contradicting_paths / max(1.0, supporting_paths + contradicting_paths)
    request_ids = {
        "scenario_id": "LuaHistory_2026-06-23_first_contact",
        "contact_id": obs.observation_id,
        "observation_time": str(obs.time),
        "hypothesis": candidate["hypothesis"],
    }
    features = ContactHypothesisFeatures(
        **request_ids,
        supporting_path_count=supporting_paths,
        contradicting_path_count=contradicting_paths,
        mean_source_reliability=0.75,
        recency=1.0 / (1.0 + float(obs.emission_age or 0.0)),
        shortest_path_to_platform_class=1.0 if class_match else 3.0,
        emission_match_score=emission_match,
        kinematic_match_score=kinematic_match,
        contradiction_score=contradiction_score,
        evidence_query_id=evidence_query_id(**request_ids),
    )
    record = features.to_record()
    record["operator_nation"] = candidate["operator_nation"]
    record["feature_logit"] = feature_logit(features)
    record["evidence_summary"] = {
        "sensor_name": obs.emission_sensor_name,
        "matched_aliases": [a for a in candidate["emitter_aliases"] if a.lower() in obs.emission_sensor_name.lower()],
        "class_match": bool(class_match),
        "speed_kt": obs.emission_speed,
        "altitude_m": obs.emission_altitude,
    }
    return record

feature_rows = [offline_features_for_candidate(observation, candidate) for candidate in candidate_hypotheses]
features_jsonl = WORK_DIR / "first_contact_feature_rows.jsonl"
features_jsonl.write_text("".join(json.dumps(row, sort_keys=True) + "\n" for row in feature_rows), encoding="utf-8")

print(json.dumps(feature_rows, indent=2))
print(f"Wrote {features_jsonl.relative_to(REPO_ROOT)}")



## 5. Probabilistic platform and operator-nation assignment

The probability model groups feature rows for the same contact/time, converts each row to a logit, applies a softmax baseline (or a fitted temperature calibrator if provided), and marginalizes platform probabilities into operator-nation probabilities.


In [ ]:
from combat_id_calibration.probability_model import run_probability_model

probabilities_jsonl = WORK_DIR / "first_contact_probability_assignment.jsonl"
assignments = run_probability_model(features_jsonl, probabilities_jsonl)
assignment = assignments[0]

print(json.dumps(assignment, indent=2))
print(f"Wrote {probabilities_jsonl.relative_to(REPO_ROOT)}")


## 6. LLM explanation-layer payload

This step uses the repository LLM explanation layer. It prepares a deterministic explanation payload and prompt from the probability record plus evidence summaries. The LLM layer is intentionally downstream of the probability model: it explains supplied probabilities and evidence without recalculating or changing them.


In [ ]:
from combat_id_calibration.llm_explainer import build_explanation_payload

best_platform = assignment["top_platform"]
supporting_evidence = []
contradicting_evidence = []
missing_evidence = [
    "Run the same contact against a populated Neo4j reference graph for independent radar/platform evidence paths.",
    "Add calibrated training data before using the softmax baseline operationally.",
    "Collect follow-up emissions, IFF, bearing/range history, and track-quality evidence to break ties between close MiG-29 variants.",
]

for row in feature_rows:
    summary = row["evidence_summary"]
    evidence_item = {
        "text": (
            f"{row['hypothesis']} matched aliases {summary['matched_aliases']} from "
            f"sensor {summary['sensor_name']}; class_match={summary['class_match']}; "
            f"speed_kt={summary['speed_kt']}; altitude_m={summary['altitude_m']}"
        ),
        "evidence_query_id": row["evidence_query_id"],
    }
    if row["hypothesis"] == best_platform or summary["matched_aliases"]:
        supporting_evidence.append(evidence_item)
    else:
        contradicting_evidence.append(evidence_item)

explanation_payload = build_explanation_payload(
    assignment,
    {
        "supporting_evidence": supporting_evidence,
        "contradicting_evidence": contradicting_evidence,
        "missing_evidence": missing_evidence,
    },
)

explanations_jsonl = WORK_DIR / "first_contact_explanation_payload.jsonl"
explanations_jsonl.write_text(json.dumps(explanation_payload, sort_keys=True) + "\n", encoding="utf-8")

print(json.dumps(explanation_payload, indent=2))
print(f"Wrote {explanations_jsonl.relative_to(REPO_ROOT)}")


## 7. Human-readable conclusion

The result below should be treated as a development baseline unless a fitted calibration model and richer Neo4j evidence graph are supplied. The notebook preserves the candidate distribution and explanation payload rather than only the winning label.


In [ ]:
print(
    f"Top emitter platform type: {assignment['top_platform']} "
    f"({assignment['top_platform_probability']:.1%})"
)
print(
    f"Top operator nation: {assignment['top_operator_nation']} "
    f"({assignment['top_operator_nation_probability']:.1%})"
)
print("\nPlatform distribution:")
for platform, probability in assignment["platform_probabilities"].items():
    print(f"  - {platform}: {probability:.1%}")
print("\nOperator-nation distribution:")
for nation, probability in assignment["operator_nation_probabilities"].items():
    print(f"  - {nation}: {probability:.1%}")
